# 00 Data Exploration — M5 Retail Demand

This notebook is the first exploratory pass for the retail replenishment and allocation project. The goal is to understand the M5 Walmart dataset before moving into forecasting, inventory logic, or allocation rules.

Main objectives:
1. Understand how `Y_df`, `X_df`, and `S_df` fit together.
2. Audit the sales history and retail hierarchy.
3. Build clear category-level daily views for trend, seasonality, and low-sales diagnostics.
4. Keep the analysis portfolio-friendly and easy to extend later.


## 1. Setup

This notebook keeps the setup block intentionally thin. Reusable plotting, path resolution, and calendar helpers live in `helpers/eda.py` so they can be reused in later feature engineering notebooks.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = None
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "pyproject.toml").exists() and (candidate / "helpers").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root from this notebook.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from helpers.eda import (
    add_calendar_flags,
    bootstrap_notebook,
    finish_axis,
    plot_category_bar_panels,
    plot_category_histograms,
    plot_category_multiline,
    plot_category_panels,
)

notebook_context = bootstrap_notebook(PROJECT_ROOT)
DATA_DIR = notebook_context.data_dir
PROCESSED_DIR = notebook_context.processed_dir
category_colors = notebook_context.category_colors


## 2. Load M5 data

`datasetsforecast` returns three related tables:

- `Y_df`: the target sales history
- `X_df`: time-varying features such as sell price, events, and SNAP indicators
- `S_df`: static item/store/category metadata

We will inspect them separately first, then join them into one analytical table called `m5`.


In [ ]:
from datasetsforecast.m5 import M5

Y_df, X_df, S_df = M5.load(directory=DATA_DIR, cache=True)

print("Y_df:", Y_df.shape)
print("X_df:", X_df.shape)
print("S_df:", S_df.shape)


### Quick look at the three source tables


In [ ]:
Y_df.head()


In [ ]:
X_df.head()


In [ ]:
S_df.head()


## 3. Initial schema and data quality audit

Before aggregating anything, confirm the basic structure of each table and check for obvious target issues such as duplicates, missing values, or negative sales.


In [ ]:
for name, df in {"Y_df": Y_df, "X_df": X_df, "S_df": S_df}.items():
    print(f"\n{name}")
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())
    print(df.dtypes)


In [ ]:
print("Date range:", Y_df["ds"].min(), "to", Y_df["ds"].max())
print("Number of unique series:", Y_df["unique_id"].nunique())
print("Rows:", len(Y_df))
print("Duplicate unique_id-date rows:", Y_df.duplicated(subset=["unique_id", "ds"]).sum())
print("Negative sales rows:", (Y_df["y"] < 0).sum())
print("Missing target values:", Y_df["y"].isna().sum())


## 4. Create the main joined table

`m5` is the main SKU-store-day table used through the rest of the notebook. It combines:

- target sales from `Y_df`
- time-varying attributes from `X_df`
- static hierarchy attributes from `S_df`


In [ ]:
# Join sales, time-varying features, and static hierarchy into one analytical table.
m5 = (
    Y_df
    .merge(X_df, on=["unique_id", "ds"], how="left")
    .merge(S_df, on="unique_id", how="left")
)

print("m5 shape:", m5.shape)
m5.head()


In [ ]:
missing_summary = (
    m5.isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_rate")
    .reset_index()
    .rename(columns={"index": "column"})
)

missing_summary.head(25)


In [ ]:
series_lengths = (
    m5.groupby("unique_id", observed=True)["ds"]
    .agg(start_date="min", end_date="max", observations="count")
    .reset_index()
)

series_lengths.describe(include="all")


## 5. Hierarchy exploration

The retail structure matters because this project will eventually move between SKU-store decisions and higher-level category signals. This section maps the hierarchy first.


In [ ]:
hierarchy_cols = ["state_id", "store_id", "cat_id", "dept_id", "item_id"]

S_df[hierarchy_cols].nunique()


In [ ]:
unique_values = {
    col: sorted(S_df[col].dropna().unique().tolist())
    for col in hierarchy_cols
}

# item_id is long, so preview only the first 20 values.
for col, values in unique_values.items():
    print(f"\n{col}: {len(values)} unique")
    print(values[:20])


In [ ]:
store_category_structure = (
    S_df
    .groupby(["state_id", "store_id", "cat_id"], observed=True)
    .agg(
        item_count=("item_id", "nunique"),
        series_count=("unique_id", "nunique"),
    )
    .reset_index()
    .sort_values(["state_id", "store_id", "cat_id"])
)

store_category_structure


In [ ]:
hierarchy_summary = (
    S_df
    .groupby(["state_id", "store_id", "cat_id", "dept_id"], observed=True)
    .agg(series_count=("unique_id", "nunique"))
    .reset_index()
    .sort_values(["state_id", "store_id", "cat_id", "dept_id"])
)

hierarchy_summary.head(25)


## 6. Build the core category-level views

We keep three related category-level tables throughout the notebook:

- `category_daily_raw`: the source-of-truth category-day sales table with all dates retained
- `category_daily_event_adjusted`: the same sales totals enriched with event and calendar flags for diagnostics
- `category_daily_normalized`: a diagnostic-only view that excludes December 25 so one extreme holiday does not dominate certain charts

Important: December 25 is **not** deleted from the raw data. It stays in the raw and event-aware views. It is only excluded from the normalized diagnostic view.


In [ ]:
# Raw category view: this keeps every observed category-day exactly as loaded.
category_daily_raw = (
    m5
    .groupby(["ds", "cat_id"], observed=True)
    .agg(y=("y", "sum"))
    .reset_index()
    .sort_values(["cat_id", "ds"])
)

# Event columns are date-level, so deduplicate them before joining onto category totals.
calendar_daily = (
    X_df[["ds", "event_name_1", "event_type_1", "event_name_2", "event_type_2"]]
    .drop_duplicates()
    .sort_values("ds")
)

assert calendar_daily["ds"].is_unique, "Expected one calendar row per date."

# Event-adjusted means calendar-aware for diagnostics; sales values are not mathematically altered here.
category_daily_event_adjusted = add_calendar_flags(
    category_daily_raw.merge(calendar_daily, on="ds", how="left")
).sort_values(["cat_id", "ds"])

# Normalized view is diagnostic only: exclude Dec 25 to inspect typical demand without one extreme holiday dominating.
category_daily_normalized = category_daily_event_adjusted.loc[
    ~category_daily_event_adjusted["is_christmas"]
].copy()

view_summary = pd.DataFrame(
    [
         {
             "view_name": "category_daily_raw",
             "rows": len(category_daily_raw),
            "includes_dec_25": (
                category_daily_raw["ds"].dt.month.eq(12)
                & category_daily_raw["ds"].dt.day.eq(25)
            ).any(),
             "purpose": "Source-of-truth category sales history",
         },
        {
            "view_name": "category_daily_event_adjusted",
            "rows": len(category_daily_event_adjusted),
            "includes_dec_25": category_daily_event_adjusted["is_christmas"].any(),
            "purpose": "Same sales totals plus event and calendar flags",
        },
        {
            "view_name": "category_daily_normalized",
            "rows": len(category_daily_normalized),
            "includes_dec_25": category_daily_normalized["is_christmas"].any(),
            "purpose": "Diagnostic view excluding Dec 25 only",
        },
    ]
)

view_summary


In [ ]:
# These lower-level aggregates will be useful later for supply-chain diagnostics.
store_category_daily = (
    m5
    .groupby(["ds", "state_id", "store_id", "cat_id"], observed=True)
    .agg(y=("y", "sum"))
    .reset_index()
    .sort_values(["state_id", "store_id", "cat_id", "ds"])
)

state_category_daily = (
    m5
    .groupby(["ds", "state_id", "cat_id"], observed=True)
    .agg(y=("y", "sum"))
    .reset_index()
    .sort_values(["state_id", "cat_id", "ds"])
)

store_category_daily.head()


## 7. Total and category-level sales trends

Start with the most readable business view: total daily sales and category-level daily sales. This gives a first sense of overall scale, trend smoothness, and whether one category dominates the signal.


In [ ]:
daily_sales = (
    category_daily_raw
    .groupby("ds", observed=True)
    .agg(y=("y", "sum"))
    .reset_index()
)

ax = daily_sales.plot(x="ds", y="y", figsize=(16, 6), legend=False, color="tab:blue")
finish_axis(ax, title="Total Daily Unit Sales", xlabel="Date", ylabel="Units Sold")
plt.show()


In [ ]:
plot_category_multiline(category_daily_raw, title="Daily Sales by Category | Raw View")


In [ ]:
plot_category_panels(category_daily_raw, title_suffix="Raw View")


In [ ]:
plot_category_multiline(category_daily_normalized, title="Daily Sales by Category | Normalized Diagnostic View (Dec 25 Excluded Only)")


## 8. Extreme low-sales days and holiday diagnostics

This section asks whether unusually low category days look like data errors, predictable holiday behavior, or narrower store-level effects. The goal is diagnosis, not a final causal claim.


In [ ]:
zero_category_days = category_daily_raw.loc[category_daily_raw["y"].eq(0)].copy()

print("Zero category-day observations:", len(zero_category_days))
zero_category_days.sort_values(["ds", "cat_id"])


In [ ]:
low_sales_days = (
    category_daily_event_adjusted
    .assign(
        p01_by_category=lambda df: (
            df.groupby("cat_id", observed=True)["y"]
            .transform(lambda s: s.quantile(0.01))
        )
    )
    .loc[lambda df: df["y"] <= df["p01_by_category"]]
    .sort_values(["cat_id", "y", "ds"])
)

low_sales_days


In [ ]:
christmas_impact = (
    category_daily_event_adjusted
    .groupby(["cat_id", "is_christmas"], observed=True)["y"]
    .agg(count="count", mean="mean", median="median", min="min", max="max")
    .reset_index()
)

christmas_impact_pivot = (
    christmas_impact
    .pivot(index="cat_id", columns="is_christmas", values="mean")
    .rename(columns={False: "normal_avg_sales", True: "christmas_avg_sales"})
)

christmas_impact_pivot["christmas_sales_retention"] = (
    christmas_impact_pivot["christmas_avg_sales"] / christmas_impact_pivot["normal_avg_sales"]
)
christmas_impact_pivot["christmas_sales_drop_pct"] = 1 - christmas_impact_pivot["christmas_sales_retention"]

christmas_impact_pivot.round(3)


In [ ]:
# Drill deeper into Christmas to avoid assuming that all stores are simply "closed."
christmas_detail = m5.loc[
    m5["ds"].dt.month.eq(12)
    & m5["ds"].dt.day.eq(25)
    & m5["y"].gt(0)
].copy()

christmas_store_sales = (
    christmas_detail
    .assign(year=lambda df: df["ds"].dt.year)
    .groupby(["year", "cat_id", "state_id", "store_id"], observed=True)
    .agg(
        total_sales=("y", "sum"),
        selling_items=("item_id", "nunique"),
        selling_series=("unique_id", "nunique"),
    )
    .reset_index()
    .sort_values(["year", "cat_id", "total_sales"], ascending=[True, True, False])
)

christmas_store_sales.head(20)


In [ ]:
christmas_category_summary = (
    christmas_store_sales
    .groupby(["year", "cat_id"], observed=True)
    .agg(
        stores_with_sales=("store_id", "nunique"),
        total_christmas_sales=("total_sales", "sum"),
        total_selling_items=("selling_items", "sum"),
        total_selling_series=("selling_series", "sum"),
    )
    .reset_index()
)

christmas_category_summary


In [ ]:
thanksgiving_impact = (
    category_daily_event_adjusted
    .groupby(["cat_id", "is_thanksgiving_window"], observed=True)["y"]
    .agg(count="count", mean="mean", median="median", min="min", max="max")
    .reset_index()
)

thanksgiving_impact


### Working interpretation

December 25 should not automatically be treated as bad data or confirmed store closure. The raw data keeps those observations, and some sales still appear on that date. For this notebook, December 25 is excluded only in `category_daily_normalized`, which is a diagnostic view for studying typical demand without one extreme holiday dominating the scale.


## 9. Sales distribution analysis

Distribution views help show how different the categories are in scale and spread. The raw view reflects real operating behavior, while the normalized view helps us inspect the shape of ordinary demand after removing December 25 from that one diagnostic slice.


In [ ]:
category_summary_raw = (
    category_daily_raw
    .groupby("cat_id", observed=True)["y"]
    .agg(count="count", mean="mean", median="median", std="std", skew="skew", min="min", max="max")
    .round(2)
)

category_summary_raw


In [ ]:
category_summary_normalized = (
    category_daily_normalized
    .groupby("cat_id", observed=True)["y"]
    .agg(count="count", mean="mean", median="median", std="std", skew="skew", min="min", max="max")
    .round(2)
)

category_summary_normalized


In [ ]:
plot_category_histograms(category_daily_raw, title_suffix="Raw View", density=False)


In [ ]:
plot_category_histograms(category_daily_normalized, title_suffix="Normalized View (Dec 25 Excluded)", density=False)


In [ ]:
overall_mean = category_daily_normalized["y"].mean()
overall_median = category_daily_normalized["y"].median()

plt.figure(figsize=(14, 6))

for cat in sorted(category_daily_normalized["cat_id"].unique()):
    data = category_daily_normalized.loc[category_daily_normalized["cat_id"].eq(cat), "y"]
    plt.hist(
        data,
        bins="sqrt",
        density=True,
        alpha=0.45,
        edgecolor="black",
        label=cat,
        color=category_colors.get(cat, "tab:gray"),
    )

plt.axvline(overall_mean, color="black", linestyle="--", label=f"Overall mean: {overall_mean:,.0f}")
plt.axvline(overall_median, color="red", linestyle=":", label=f"Overall median: {overall_median:,.0f}")
plt.title("Category Sales Density | Normalized Diagnostic View")
plt.xlabel("Daily Units Sold")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()


### Distribution interpretation

The category distributions are not interchangeable. `FOODS` operates at a much larger daily scale than `HOBBIES` or `HOUSEHOLD`, so category-level normalization and category-specific evaluation will matter later.


## 10. Seasonality diagnostics

Seasonality is more useful when viewed by category instead of only in the aggregate. This prevents the highest-volume category from dominating the interpretation.


In [ ]:
seasonality_df = add_calendar_flags(category_daily_raw)
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

category_dow_sales = (
    seasonality_df
    .groupby(["cat_id", "dayofweek"], observed=True)["y"]
    .mean()
    .reset_index()
)
category_dow_sales["dayofweek"] = pd.Categorical(
    category_dow_sales["dayofweek"],
    categories=weekday_order,
    ordered=True,
)
category_dow_sales = category_dow_sales.sort_values(["cat_id", "dayofweek"])

category_dow_sales


In [ ]:
plot_category_bar_panels(
    category_dow_sales,
    x_col="dayofweek",
    y_col="y",
    title_prefix="Average Daily Sales by Day of Week",
    xlabel="Day of Week",
    ylabel="Average Units Sold",
    rotation=45,
)


In [ ]:
category_monthly_sales = (
    seasonality_df
    .groupby(["cat_id", "month"], observed=True)["y"]
    .mean()
    .reset_index()
)

plot_category_bar_panels(
    category_monthly_sales,
    x_col="month",
    y_col="y",
    title_prefix="Average Daily Sales by Month",
    xlabel="Month",
    ylabel="Average Units Sold",
)


## 11. Event and SNAP first pass

This section is intentionally light-touch. We only want a first look at event and SNAP context now. Deeper interpretation belongs in feature engineering, where we can evaluate whether these fields improve forecasts or inventory decisions.


In [ ]:
event_sales = (
    m5
    .groupby(["cat_id", "event_type_1"], observed=True)["y"]
    .agg(mean="mean", median="median", count="count")
    .reset_index()
    .sort_values(["cat_id", "mean"], ascending=[True, False])
)

event_sales


In [ ]:
snap_cols = ["snap_CA", "snap_TX", "snap_WI"]

snap_effects = []
for state, snap_col in zip(["CA", "TX", "WI"], snap_cols):
    tmp = (
        m5.loc[m5["state_id"].eq(state)]
        .groupby(["cat_id", snap_col], observed=True)["y"]
        .agg(mean="mean", median="median", count="count")
        .reset_index()
        .rename(columns={snap_col: "snap_flag"})
    )
    tmp["state_id"] = state
    tmp["snap_col"] = snap_col
    snap_effects.append(tmp)

snap_effects = pd.concat(snap_effects, ignore_index=True)
snap_effects.sort_values(["state_id", "cat_id", "snap_flag"])


## 12. SKU-store intermittency and demand profile

Category demand can look smooth even when the underlying SKU-store series are sparse. This matters for retail replenishment because operational decisions happen closer to the series level than the category level.


In [ ]:
series_profile = (
    m5
    .groupby("unique_id", observed=True)
    .agg(
        avg_daily_sales=("y", "mean"),
        median_daily_sales=("y", "median"),
        total_sales=("y", "sum"),
        zero_sales_rate=("y", lambda s: (s == 0).mean()),
        sales_std=("y", "std"),
        days_observed=("y", "count"),
    )
    .reset_index()
    .merge(S_df, on="unique_id", how="left")
)

series_profile.head()


In [ ]:
def demand_type(row):
    if row["zero_sales_rate"] >= 0.70:
        return "Very intermittent"
    if row["zero_sales_rate"] >= 0.40:
        return "Intermittent"
    if row["avg_daily_sales"] >= 5:
        return "High volume"
    return "Regular / low volume"


series_profile["demand_type"] = series_profile.apply(demand_type, axis=1)

series_profile["demand_type"].value_counts()


In [ ]:
series_profile_by_category = (
    series_profile
    .groupby(["cat_id", "demand_type"], observed=True)
    .agg(
        series_count=("unique_id", "nunique"),
        avg_zero_rate=("zero_sales_rate", "mean"),
        avg_daily_sales=("avg_daily_sales", "mean"),
    )
    .reset_index()
    .sort_values(["cat_id", "series_count"], ascending=[True, False])
)

series_profile_by_category


## 13. Price behavior

Price is part of the exogenous signal. At this stage, we only want a descriptive view of how much price movement exists and what one high-volume series looks like over time.


In [ ]:
price_profile = (
    m5
    .groupby("unique_id", observed=True)
    .agg(
        avg_price=("sell_price", "mean"),
        min_price=("sell_price", "min"),
        max_price=("sell_price", "max"),
        price_changes=("sell_price", "nunique"),
        total_sales=("y", "sum"),
    )
    .reset_index()
    .merge(S_df, on="unique_id", how="left")
)

price_profile.sort_values("price_changes", ascending=False).head(20)


In [ ]:
sample_id = series_profile.sort_values("total_sales", ascending=False)["unique_id"].iloc[0]
sample = m5.loc[m5["unique_id"].eq(sample_id)].copy().sort_values("ds")

ax = sample.plot(
    x="ds",
    y=["y", "sell_price"],
    figsize=(16, 6),
    title=f"Sales and Price - {sample_id}",
)
finish_axis(ax, title=f"Sales and Price - {sample_id}", xlabel="Date", ylabel="Value")
plt.show()


## 14. Simple baseline forecasting sanity check

This is still exploratory. The point is not to optimize a model yet, but to get a rough sense of how predictable category-level demand looks under very simple benchmarks.


In [ ]:
H = 28


def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))


def wape(y_true, y_pred):
    denom = np.sum(np.abs(y_true))
    return np.nan if denom == 0 else np.sum(np.abs(y_true - y_pred)) / denom


def evaluate_baselines_one_series(df: pd.DataFrame, h: int = 28) -> list[dict]:
    df = df.sort_values("ds")
    train = df.iloc[:-h].copy()
    test = df.iloc[-h:].copy()

    forecasts = {
        "naive_last": train["y"].iloc[-1],
        "ma_28": train["y"].tail(28).mean(),
        "ma_56": train["y"].tail(56).mean(),
    }

    rows = []
    for model_name, pred in forecasts.items():
        y_pred = np.repeat(pred, len(test))
        rows.append(
            {
                "model": model_name,
                "mae": mae(test["y"].to_numpy(), y_pred),
                "wape": wape(test["y"].to_numpy(), y_pred),
            }
        )
    return rows


In [ ]:
baseline_results = []

for cat in sorted(category_daily_raw["cat_id"].unique()):
    df = category_daily_raw.loc[category_daily_raw["cat_id"].eq(cat), ["ds", "y"]].copy()
    for row in evaluate_baselines_one_series(df, h=H):
        row["cat_id"] = cat
        baseline_results.append(row)

baseline_results = pd.DataFrame(baseline_results)
baseline_results.sort_values(["cat_id", "wape"])


In [ ]:
cat = "FOODS"
df = category_daily_raw.loc[category_daily_raw["cat_id"].eq(cat), ["ds", "y"]].sort_values("ds").copy()
train = df.iloc[:-H].copy()
test = df.iloc[-H:].copy()

plot_df = test.copy()
plot_df["naive_last"] = train["y"].iloc[-1]
plot_df["ma_28"] = train["y"].tail(28).mean()
plot_df["ma_56"] = train["y"].tail(56).mean()

ax = plot_df.plot(x="ds", y=["y", "naive_last", "ma_28", "ma_56"], figsize=(14, 5))
finish_axis(ax, title=f"28-Day Baseline Forecast Check - {cat}", xlabel="Date", ylabel="Units Sold")
plt.show()


## 15. Save processed analytical tables

These outputs make the next notebooks easier to build because they preserve the main EDA-level aggregates without repeating the joins and groupbys each time.


In [ ]:
category_daily_raw.to_parquet(PROCESSED_DIR / "category_daily_raw.parquet", index=False)
category_daily_event_adjusted.to_parquet(
    PROCESSED_DIR / "category_daily_event_adjusted.parquet",
    index=False,
)
category_daily_normalized.to_parquet(
    PROCESSED_DIR / "category_daily_normalized_excluding_christmas.parquet",
    index=False,
)
store_category_daily.to_parquet(PROCESSED_DIR / "store_category_daily.parquet", index=False)
state_category_daily.to_parquet(PROCESSED_DIR / "state_category_daily.parquet", index=False)
series_profile.to_parquet(PROCESSED_DIR / "series_profile.parquet", index=False)

print("Saved processed tables to", PROCESSED_DIR)


## 16. Current EDA Conclusions

Preliminary conclusions from this first pass:

1. The three M5 tables join cleanly, and this EDA pass found no duplicate `unique_id`-date rows, no negative sales, and no missing target values in `Y_df`.
2. Category demand operates at clearly different scales, with `FOODS` far above `HOUSEHOLD`, and `HOUSEHOLD` above `HOBBIES`, so later evaluation should stay category-aware.
3. December 25 behaves like an extreme calendar effect with near-zero category sales, but it remains in the raw data; only the normalized diagnostic view excludes it.
4. Category-level demand is smoother than the underlying SKU-store series, while many SKU-store histories still show intermittent behavior.
5. Event and SNAP fields deserve a more careful pass in feature engineering, but this notebook stops at a descriptive first look rather than making causal claims.
